In [0]:
import json, random, uuid
from datetime import datetime, timedelta, timezone

CATALOG, SCHEMA = "payment_lakehouse", "pipeline"
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"

REGIONS = [
    {"code":"IN","weight":0.40,"currency":"INR","retry_window_days":7},
    {"code":"US","weight":0.20,"currency":"USD","retry_window_days":3},
    {"code":"EU","weight":0.15,"currency":"EUR","retry_window_days":5},
    {"code":"SG","weight":0.15,"currency":"SGD","retry_window_days":4},
    {"code":"AE","weight":0.10,"currency":"AED","retry_window_days":10},
]
DEFECTS = {"duplicate":0.02,"malformed":0.01,"late_invoice":0.05,
           "amount_mismatch":0.03,"no_invoice":0.08}
METHODS = ["WIRE","ACH","CARD","SEPA"]

In [0]:
def iso(ts): return ts.astimezone(timezone.utc).isoformat().replace("+00:00","Z")

CORRUPTIONS = [
    ("null_amount",    lambda e: e.update(amount=None)),
    ("string_amount",  lambda e: e.update(amount="not-a-number")),
    ("bad_currency",   lambda e: e.update(currency="XXX")),
    ("missing_region", lambda e: e.pop("region", None)),
    ("negative_amount",lambda e: e.update(amount=-abs(e.get("amount") or 1))),
]

def generate(n=5000, seed=42):
    rng, now = random.Random(seed), datetime.now(timezone.utc)
    window = {r["code"]: r["retry_window_days"] for r in REGIONS}
    events, truth = [], {k:0 for k in
        ["payments_generated","MATCHED","AMOUNT_MISMATCH","PENDING","EXPIRED","REJECTED"]}

    for i in range(n):
        region = rng.choices(REGIONS, weights=[r["weight"] for r in REGIONS])[0]
        p_ts   = now - timedelta(days=rng.uniform(0, 12))
        pid    = f"PAY-{i:07d}"
        amount = round(rng.uniform(50, 5000), 2)

        pay = {"event_id":str(uuid.uuid4()), "event_type":"PAYMENT_RECEIVED",
               "event_ts":iso(p_ts), "payment_id":pid,
               "customer_id":f"CUST-{rng.randint(1,20000):06d}",
               "region":region["code"], "currency":region["currency"],
               "amount":amount, "payment_method":rng.choice(METHODS)}
        truth["payments_generated"] += 1

        if rng.random() < DEFECTS["malformed"]:
            rng.choice(CORRUPTIONS)[1](pay)
            truth["REJECTED"] += 1
            events.append(pay); continue
        events.append(pay)

        if rng.random() < DEFECTS["no_invoice"]:
            inv_ts = None
        elif rng.random() < DEFECTS["late_invoice"]:
            inv_ts = p_ts + timedelta(days=rng.uniform(1, 7))
        else:
            inv_ts = p_ts + timedelta(hours=rng.uniform(0, 24))

        if inv_ts and inv_ts <= now:
            mismatch = rng.random() < DEFECTS["amount_mismatch"]
            events.append({"event_id":str(uuid.uuid4()), "event_type":"INVOICE_CLEARED",
                "event_ts":iso(inv_ts), "invoice_id":f"INV-{i:07d}", "payment_id":pid,
                "region":region["code"], "currency":region["currency"],
                "amount":round(amount*rng.uniform(0.80,0.98),2) if mismatch else amount})
            truth["AMOUNT_MISMATCH" if mismatch else "MATCHED"] += 1
        else:
            days = (now - p_ts).days
            truth["EXPIRED" if days > window[region["code"]] else "PENDING"] += 1

    dupes = 0
    for e in list(events):
        if rng.random() < DEFECTS["duplicate"]:
            events.append(dict(e)); dupes += 1

    events.sort(key=lambda e: e["event_ts"])
    truth.update(duplicates_injected=dupes, events_emitted=len(events),
                 generated_at=iso(now), seed=seed)
    return events, truth

In [0]:
import os
events, truth = generate(5000, 42)

os.makedirs(f"{LANDING}/events", exist_ok=True)
for i in range(0, len(events), 100):
    with open(f"{LANDING}/events/part-{i//100:04d}.json", "w") as f:
        f.write("\n".join(json.dumps(e) for e in events[i:i+100]))

with open(f"{LANDING}/_expected_totals.json", "w") as f:
    json.dump(truth, f, indent=2)

print(f"{len(events):,} events -> {len(events)//100 + 1} files")
for k, v in truth.items(): print(f"  {k:<22}: {v}")

In [0]:
display(dbutils.fs.ls(f"{LANDING}/events")[:5])